[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_18/MFM_ch18_bank_rate_risk/MFM_ch18_bank_rate_risk.ipynb)

# MFM_ch18_bank_rate_risk

Case study of Jiang, Matvos, Piskorski and Seru (2024), mark-to-market losses and US bank fragility in 2023, on the course data: the mark-to-market loss of two US Treasury ETFs (IEF, 7-10 years; TLT, 20+ years) from 31 December 2021 to 31 March 2023, and the rolling 252-day rate beta of the six US banks of the chapter (equal weights) on the daily change of the US 10-year yield, given the S&P 500, with 95% Newey-West intervals.

*Modelling Financial Markets (MFM), Chapter 18: Systemic Risk and Networks. Author: Daniel Traian Pele.*

Paper: Jiang, E.X., Matvos, G., Piskorski, T., Seru, A. (2024). Monetary tightening and U.S. bank fragility in 2023: mark-to-market losses and uninsured depositor runs? *Journal of Financial Economics* 159, 103899, [doi:10.1016/j.jfineco.2024.103899](https://doi.org/10.1016/j.jfineco.2024.103899).

The paper marks bank balance sheets to market; this notebook uses market prices only. The rate beta $b_Y$ comes from $r_{B,t} = a + b_M r_{M,t} + b_Y \Delta y_t + \varepsilon_t$ on rolling 252-day windows, with $\Delta y_t$ the daily change of the 10-year yield in basis points, reported per +100 bp. The Newey-West intervals require a finite fourth moment.

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_18'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

## Style

In [ ]:
plt.rcParams.update({'figure.facecolor': 'none', 'axes.facecolor': 'none', 'savefig.facecolor': 'none',
                     'savefig.transparent': True, 'axes.grid': False, 'font.family': 'sans-serif',
                     'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'], 'font.size': 9, 'axes.labelsize': 10,
                     'axes.titlesize': 11, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.linewidth': 0.6, 'lines.linewidth': 1.1, 'legend.facecolor': 'none',
                     'legend.framealpha': 0, 'legend.fontsize': 8})

# course colours (generate_all_charts.py)
MainBlue, IDAred, Forest, Amber, Purple, Navy, BandBlue = ('#1A3A6E', '#CD0000', '#2E7D32', '#B5853F', '#8E44AD',
                                                           '#1F2A44', '#C5D2E8')

## Data

Daily prices from the course repository (data/market): the six US banks, the S&P 500, the IEF and TLT Treasury ETFs and the US 2-year and 10-year yields.

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
START = '2010-01-01'
END = '2026-09-18'

# the six US banks of the chapter: key -> (symbol, name, region)
BANKS = {
    'JPM':  ('JPM.US', 'JPMorgan Chase',  'US'),
    'BAC':  ('BAC.US', 'Bank of America', 'US'),
    'C':    ('C.US',   'Citigroup',       'US'),
    'GS':   ('GS.US',  'Goldman Sachs',   'US'),
    'MS':   ('MS.US',  'Morgan Stanley',  'US'),
    'WFC':  ('WFC.US', 'Wells Fargo',     'US'),
}
US = list(BANKS)
INDICES = {'SPX': ('GSPC.INDX', 'S&P 500')}


def read_market(symbol):
    """Read the daily price series of one symbol from the course data."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def bank_frame(key, start=START, end=END):
    """Daily prices and volume of one bank, with the adjusted price and the corrections above."""
    symbol, _, region = BANKS[key]
    d = read_market(symbol).loc[start:end].copy()
    d = d[(d.index.dayofweek < 5) & (d['adjusted_close'] > 0)]
    if region == 'RO':
        d = d[d['volume'] > 0]                                   # days without trades
    if key == 'TLV' and pd.Timestamp('2016-05-30') in d.index:
        # bonus-share adjustment: the post-ex-date factor also applies on 30 May 2016
        f = d.loc['2016-05-31', 'adjusted_close'] / d.loc['2016-05-31', 'close']
        d.loc['2016-05-30', 'adjusted_close'] = d.loc['2016-05-30', 'close'] * f
    return d


def bank_price(key, start=START, end=END):
    return bank_frame(key, start, end)['adjusted_close'].rename(key)


def artificial_fill(d):
    """Rows that repeat the previous close and show no trading: an artificial non-trading-day fill.

    A repeated close is a fill only if the row itself shows no trading: zero volume (for files that report
    volume), no intraday range (open = high = low = close) or a copy of the previous row in every field. A
    repeated close on a day with volume and an intraday range is a genuine unchanged close (a zero return)
    and is kept. Files with closes only (the BET) give no such evidence, so every row is kept.
    """
    repeat = d['close'].diff() == 0
    no_trade = pd.Series(False, index=d.index)
    if 'volume' in d and (d['volume'] > 0).mean() > 0.5:
        no_trade |= d['volume'].fillna(0) <= 0
    if {'open', 'high', 'low'} <= set(d.columns):
        no_trade |= (d['open'] == d['high']) & (d['high'] == d['low']) & (d['low'] == d['close'])
        fields = [c for c in ['open', 'high', 'low', 'close', 'volume'] if c in d]
        no_trade |= (d[fields] == d[fields].shift()).all(axis=1)
    return repeat & no_trade


def index_close(key, start=START, end=END):
    d = read_market(INDICES[key][0]).loc[start:end]
    d = d[(d.index.dayofweek < 5) & (d['close'] > 0)].dropna(subset=['close'])
    if key != 'VIX':
        d = d[~artificial_fill(d)]                               # non-trading days filled with the previous price
    return d['close'].rename(key)


def prices(keys, start=START, end=END):
    """Daily prices of banks and/or indices, on common trading days only."""
    cols = [bank_price(k, start, end) if k in BANKS else index_close(k, start, end) for k in keys]
    return pd.concat(cols, axis=1, join='inner').dropna()

## Rate beta with Newey-West standard errors

In [ ]:
WINDOW = 252          # rolling window (trading days)
NW_LAGS = 5           # Newey-West lags
BASE, MTM_END = '2021-12-31', '2023-03-31'          # the paper's interval: Q1 2022 to Q1 2023
SVB = '2023-03-10'                                    # FDIC takes over Silicon Valley Bank
PERIODS = [('2012-2019', '2012-01-01', '2019-12-31'), ('2020-2021', '2020-01-01', '2021-12-31'),
           ('Tightening', '2022-01-01', '2023-03-09'), ('SVB to First Republic', '2023-03-10', '2023-05-31'),
           ('After', '2023-06-01', '2026-12-31')]


def load_data():
    """Common trading days of the six US banks, the S&P 500, IEF, TLT and the US 2Y and 10Y yields."""
    P = prices(US + ['SPX'])
    extra = {'IEF': read_market('IEF.US')['adjusted_close'], 'TLT': read_market('TLT.US')['adjusted_close'],
             'Y10': read_market('US10Y.GBOND')['close'], 'Y2': read_market('US2Y.GBOND')['close']}
    X = pd.concat([P] + [s.rename(k) for k, s in extra.items()], axis=1, join='inner').dropna()
    y10 = read_market('US10Y.GBOND').reindex(X.index)
    stale = (X['Y10'].diff() == 0) & (y10['open'] == y10['high']) & (y10['high'] == y10['low']) & (y10['low'] == y10['close'])
    X = X[~stale]      # a repeated 10Y quote with no intraday range: bond market closed, previous yield carried forward
    D = pd.DataFrame(index=X.index)
    D['bank'] = 100 * (np.exp(np.log(X[US]).diff()) - 1).mean(axis=1)          # equal weights, daily rebalancing
    D['spx'] = 100 * X['SPX'].pct_change()
    D['ief'] = 100 * X['IEF'].pct_change()
    D['tlt'] = 100 * X['TLT'].pct_change()
    D['d10'] = 100 * X['Y10'].diff()                                              # basis points
    D['y10'] = X['Y10']
    D['y2'] = X['Y2']
    return D.dropna()


def ols_nw(y, X, lags=NW_LAGS):
    """OLS coefficients and Newey-West (Bartlett) standard errors."""
    XtX_inv = np.linalg.inv(X.T @ X)
    b = XtX_inv @ X.T @ y
    u = y - X @ b
    g = X * u[:, None]
    S = g.T @ g
    for L in range(1, lags + 1):
        w = 1 - L / (lags + 1)
        G = g[L:].T @ g[:-L]
        S += w * (G + G.T)
    V = XtX_inv @ S @ XtX_inv
    return b, np.sqrt(np.diag(V))


def rate_beta(D, dep='bank', market=True):
    """Return (%) per +100 bp in the 10Y yield, controlling for the S&P 500 if market=True."""
    cols = ['spx', 'd10'] if market else ['d10']
    X = np.column_stack([np.ones(len(D))] + [D[c].values for c in cols])
    b, se = ols_nw(D[dep].values, X)
    return 100 * b[-1], 100 * se[-1]


def rolling_beta(D, dep='bank', market=True, window=WINDOW):
    out = []
    for t in range(window, len(D) + 1):
        b, se = rate_beta(D.iloc[t - window:t], dep, market)
        out.append((D.index[t - 1], b, se))
    return pd.DataFrame(out, columns=['date', 'b', 'se']).set_index('date')

## Chart and results

In [ ]:
def main():
    D = load_data()
    res = {'start': str(D.index[0].date()), 'end': str(D.index[-1].date()), 'n': int(len(D)), 'window': WINDOW,
           'nw_lags': NW_LAGS, 'periods': {}}
    for lab, a, b in PERIODS:
        S = D.loc[a:b]
        bb, se = rate_beta(S)
        bi, sei = rate_beta(S, 'ief', market=False)
        res['periods'][lab] = {'start': str(S.index[0].date()), 'end': str(S.index[-1].date()), 'n': int(len(S)),
                               'b': bb, 'se': se, 't': bb / se, 'b_ief': bi, 'se_ief': sei}
    # mark-to-market interval of the paper
    M = D.loc[BASE:MTM_END]
    cum = {k: 100 * ((1 + M[k].iloc[1:] / 100).prod() - 1) for k in ['bank', 'spx', 'ief', 'tlt']}
    res['mtm'] = {'base': str(M.index[0].date()), 'end': str(M.index[-1].date()), **cum,
                  'y10_0': M['y10'].iloc[0], 'y10_1': M['y10'].iloc[-1], 'y2_0': M['y2'].iloc[0], 'y2_1': M['y2'].iloc[-1]}
    # SVB week: 8 March (last close before the run) to 17 March 2023
    W = D.loc['2023-03-09':'2023-03-17']
    res['svb_week'] = {k: 100 * ((1 + W[k] / 100).prod() - 1) for k in ['bank', 'spx', 'ief']}
    res['svb_week']['d10'] = W['d10'].sum()
    res['svb_week']['d2'] = 100 * (D.loc['2023-03-17', 'y2'] - D.loc['2023-03-08', 'y2'])

    rb = rolling_beta(D)
    ri = rolling_beta(D, 'ief', market=False)
    pre = rb.loc[:'2021-12-31']
    res['roll'] = {'mean_pre2022': pre['b'].mean(), 'min_pre2022': pre['b'].min(),
                   'share_sig_pre2022': float(((pre['b'] - 1.96 * pre['se']) > 0).mean()),
                   'at_svb': float(rb.loc[:SVB, 'b'].iloc[-1]), 'at_svb_lo': float((rb.loc[:SVB, 'b'] - 1.96 * rb.loc[:SVB, 'se']).iloc[-1]),
                   'at_svb_hi': float((rb.loc[:SVB, 'b'] + 1.96 * rb.loc[:SVB, 'se']).iloc[-1]),
                   'min_2022_23': float(rb.loc['2022':'2023', 'b'].min()),
                   'min_2022_23_date': str(rb.loc['2022':'2023', 'b'].idxmin().date()),
                   'last': float(rb['b'].iloc[-1]), 'last_date': str(rb.index[-1].date()),
                   'ief_mean': float(ri['b'].mean()),
                   'first_date': str(rb.index[0].date()),
                   'min_2013_15': float(rb.loc[:'2015-12-31', 'b'].min()), 'max_2013_15': float(rb.loc[:'2015-12-31', 'b'].max()),
                   'max_2021': float(rb.loc['2021', 'b'].max()), 'max_2021_date': str(rb.loc['2021', 'b'].idxmax().date()),
                   'end_2021': float(rb.loc[:'2021-12-31', 'b'].iloc[-1])}

    # ---------------------------------------------------------------- chart
    fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.0), gridspec_kw={'width_ratios': [1, 1.25]})
    ax = axes[0]
    V = D.loc[BASE:'2023-12-31']
    for k, col, lab in [('ief', Amber, 'IEF: Treasuries 7-10y'), ('tlt', Purple, 'TLT: Treasuries 20y+'),
                        ('bank', MainBlue, 'Six US banks'), ('spx', Forest, 'S&P 500')]:
        s = 100 * (1 + V[k].iloc[1:] / 100).cumprod()
        s = pd.concat([pd.Series([100.0], index=[V.index[0]]), s])
        ax.plot(s.index, s.values, color=col, lw=1.1, label=lab)
    ax.axvspan(pd.Timestamp(BASE), pd.Timestamp(MTM_END), color=BandBlue, alpha=0.45, lw=0,
               label='Q1 2022 to Q1 2023')
    ax.axhline(100, color=Navy, lw=0.5)
    ax.axvline(pd.Timestamp(SVB), color=Navy, lw=0.6, ls=':')
    ax.text(pd.Timestamp(SVB), ax.get_ylim()[1], 'SVB', rotation=90, fontsize=7, color='black', ha='right', va='top')
    ax.set_ylabel('Value of 100 on 31 Dec 2021')
    ax.set_title('Mark-to-market: Treasuries and bank equity', fontsize=9.5)
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

    ax = axes[1]
    ax.fill_between(rb.index, rb['b'] - 1.96 * rb['se'], rb['b'] + 1.96 * rb['se'], color=BandBlue, lw=0,
                    label='Bank beta: 95% Newey-West interval')
    ax.plot(rb.index, rb['b'], color=MainBlue, lw=1.1, label='Bank beta, given the S&P 500')
    ax.plot(ri.index, ri['b'], color=Amber, lw=1.0, label='IEF beta')
    ax.axhline(0, color=Navy, lw=0.6)
    ax.axvline(pd.Timestamp('2022-03-16'), color=IDAred, lw=0.7, ls='--', label='First Fed hike')
    ax.axvline(pd.Timestamp(SVB), color=Navy, lw=0.6, ls=':')
    ax.text(pd.Timestamp(SVB), ax.get_ylim()[1], 'SVB', rotation=90, fontsize=7, color='black', ha='right', va='top')
    ax.set_ylabel('Return (%) per +100 bp\nin the US 10Y yield')
    ax.set_title(f'Rolling {WINDOW}-day rate beta', fontsize=9.5)
    ax.set_xlim(rb.index[0], rb.index[-1])
    fig.tight_layout()
    h0, l0 = axes[0].get_legend_handles_labels()
    h1, l1 = axes[1].get_legend_handles_labels()
    fig.legend(h0 + h1, l0 + l1, loc='upper center', bbox_to_anchor=(0.5, 0.0), ncol=3, frameon=False)
    os.makedirs(OUT_DIR, exist_ok=True)
    for ext, kw in [('pdf', {}), ('png', {'dpi': 180})]:
        plt.savefig(os.path.join(OUT_DIR, f'ch18_case_rate_risk.{ext}'), bbox_inches='tight', transparent=True, **kw)
    plt.show()
    plt.close()

    def js(x):
        if isinstance(x, dict):
            return {k: js(v) for k, v in x.items()}
        if isinstance(x, (np.floating, np.integer)):
            return x.item()
        return x
    with open(os.path.join(OUT_DIR, 'ch18_case_rate_risk.json'), 'w') as f:
        json.dump(js(res), f, indent=1)
    print(json.dumps(js(res), indent=1))
    return D


D = main()